# Phase 2.5 — Panel robustness checks

**Scope.** This notebook preserves the Phase 2 dataset and estimates. It adds only the pre-specified panel robustness checks for serial and cross-sectional dependence. It does not rebuild the data, perform model selection, create a dashboard, or make causal claims.

**Headline.** With country fixed effects, the original point estimates are unchanged and the food–inflation, total-confidence, and non-food-confidence associations remain distinguishable from zero under Bartlett bandwidth-3 Driscoll–Kraay uncertainty. With both country and month fixed effects, only the negative food–inflation association remains stable; total and non-food estimates materially change. The food/non-food interaction contrasts retain their direction but have wider uncertainty.

## Method and fixed specification

- Sample: Netherlands, Germany, and France; January 2016–December 2025; 360 country-month observations.
- Outcomes: year-on-year changes in Eurostat real retail-sales volume proxies (total, food, non-food).
- Predictors: headline HICP annual inflation and the Consumer Confidence Indicator.
- Model A: country fixed effects.
- Model B: country and month fixed effects.
- Covariance: Driscoll–Kraay, Bartlett kernel, bandwidth 3. No alternative bandwidths, samples, predictors, or exclusions were tried.
- Interpretation is associational. Retail measures are volume indices, not nominal household expenditure.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from phase2_5_panel_robustness import run_robustness

results = run_robustness()
panel = results["panel_table"].copy()
interaction = results["interaction_table"].copy()
print(f"Master rows: {len(results['data'])}; analytical rows per panel model: {panel['n'].unique().tolist()}")

Master rows: 396; analytical rows per panel model: [360]


## Model A — country fixed effects with Driscoll–Kraay uncertainty

In [2]:
cols = ["outcome", "term", "coefficient", "standard_error", "ci_lower_95", "ci_upper_95", "p_value", "stability_vs_original_hc3"]
display(panel.loc[panel["model"].eq("Model A: entity FE"), cols].round(4))

,outcome,term,coefficient,standard_error,ci_lower_95,ci_upper_95,p_value,stability_vs_original_hc3
0,retail_total_yoy,inflation_yoy,-0.0460,0.2211,-0.4808,0.3889,0.8354,direction and magnitude broadly stable
1,retail_total_yoy,consumer_confidence,0.1953,0.0668,0.0639,0.3267,0.0037,direction and magnitude broadly stable
4,retail_food_yoy,inflation_yoy,-0.7859,0.1646,-1.1097,-0.4621,0.0000,direction and magnitude broadly stable
5,retail_food_yoy,consumer_confidence,-0.0067,0.0396,-0.0846,0.0713,0.8663,direction and magnitude broadly stable
8,retail_nonfood_yoy,inflation_yoy,0.3214,0.3743,-0.4147,1.0575,0.3911,direction and magnitude broadly stable
9,retail_nonfood_yoy,consumer_confidence,0.3090,0.1115,0.0897,0.5282,0.0059,direction and magnitude broadly stable


Model A uses the same conditional mean specification as the Phase 2 country-fixed-effects regressions, so its coefficients match exactly. Only the covariance estimator changes. The negative food–inflation estimate remains the strongest result; confidence is positively associated with total and non-food retail growth. Total inflation, food confidence, and non-food inflation remain imprecisely estimated.

## Model B — country and month fixed effects

In [3]:
display(panel.loc[panel["model"].eq("Model B: entity + time FE"), cols].round(4))

,outcome,term,coefficient,standard_error,ci_lower_95,ci_upper_95,p_value,stability_vs_original_hc3
2,retail_total_yoy,inflation_yoy,-0.6239,0.2551,-1.1265,-0.1213,0.0152,materially changed
3,retail_total_yoy,consumer_confidence,-0.0254,0.0922,-0.2071,0.1563,0.7832,materially changed
6,retail_food_yoy,inflation_yoy,-0.6567,0.2652,-1.1792,-0.1343,0.0140,direction and magnitude broadly stable
7,retail_food_yoy,consumer_confidence,-0.0035,0.0591,-0.1199,0.1129,0.9525,direction and magnitude broadly stable
10,retail_nonfood_yoy,inflation_yoy,-0.5783,0.3733,-1.3138,0.1572,0.1227,materially changed
11,retail_nonfood_yoy,consumer_confidence,-0.0332,0.1438,-0.3165,0.2500,0.8173,materially changed


Month effects remove shocks common to all three countries. Identification therefore comes only from country deviations within each month. Under this demanding comparison, the food–inflation coefficient stays negative and similar in magnitude. The total and non-food coefficients change materially, including reversals for confidence and non-food inflation; their confidence intervals include zero except for total inflation, whose negative estimate is not stable relative to Phase 2.

## Direct coefficient comparison

In [4]:
comparison = panel[["outcome", "model", "term", "original_hc3_coefficient", "coefficient", "coefficient_change_vs_hc3", "stability_vs_original_hc3"]]
display(comparison.round(4))

,outcome,model,term,original_hc3_coefficient,coefficient,coefficient_change_vs_hc3,stability_vs_original_hc3
0,retail_total_yoy,Model A: entity FE,inflation_yoy,-0.0460,-0.0460,-0.0000,direction and magnitude broadly stable
1,retail_total_yoy,Model A: entity FE,consumer_confidence,0.1953,0.1953,-0.0000,direction and magnitude broadly stable
2,retail_total_yoy,Model B: entity + time FE,inflation_yoy,-0.0460,-0.6239,-0.5779,materially changed
3,retail_total_yoy,Model B: entity + time FE,consumer_confidence,0.1953,-0.0254,-0.2207,materially changed
4,retail_food_yoy,Model A: entity FE,inflation_yoy,-0.7859,-0.7859,0.0000,direction and magnitude broadly stable
5,retail_food_yoy,Model A: entity FE,consumer_confidence,-0.0067,-0.0067,-0.0000,direction and magnitude broadly stable
6,retail_food_yoy,Model B: entity + time FE,inflation_yoy,-0.7859,-0.6567,0.1292,direction and magnitude broadly stable
7,retail_food_yoy,Model B: entity + time FE,consumer_confidence,-0.0067,-0.0035,0.0032,direction and magnitude broadly stable
8,retail_nonfood_yoy,Model A: entity FE,inflation_yoy,0.3214,0.3214,0.0000,direction and magnitude broadly stable
9,retail_nonfood_yoy,Model A: entity FE,consumer_confidence,0.3090,0.3090,0.0000,direction and magnitude broadly stable


## Identification diagnostics

In [5]:
diag_cols = ["model", "transformed_design_rank", "transformed_condition_number", "transformed_sd_inflation", "transformed_sd_confidence", "transformed_predictor_correlation"]
display(panel[diag_cols].drop_duplicates().round(4))

,model,transformed_design_rank,transformed_condition_number,transformed_sd_inflation,transformed_sd_confidence,transformed_predictor_correlation
0,Model A: entity FE,2,3.9950,2.7624,7.5140,-0.6863
2,Model B: entity + time FE,2,3.4984,0.9588,2.7979,-0.5075


Both transformed designs have full rank (2 of 2), and condition numbers are modest. Model B has substantially less usable predictor variation after common month shocks are removed: transformed standard deviations fall from 2.76 to 0.96 for inflation and from 7.51 to 2.80 for confidence. The coefficients are identifiable, but only three countries contribute cross-country deviations in each month, so estimates with month effects should be treated as fragile rather than definitive.

## Food/non-food interaction robustness

In [6]:
display(interaction[["term", "effect_type", "coefficient", "standard_error", "ci_lower_95", "ci_upper_95", "p_value"]].round(4))

,term,effect_type,coefficient,standard_error,ci_lower_95,ci_upper_95,p_value
0,inflation_yoy:is_nonfood,interaction_parameter,0.9861,0.4104,0.1818,1.7904,0.0163
1,consumer_confidence:is_nonfood,interaction_parameter,0.2585,0.1155,0.0321,0.4848,0.0252
2,food_inflation_slope,derived_category_slope,-0.7253,0.1696,-1.0577,-0.3929,0.0000
3,nonfood_inflation_slope,derived_category_slope,0.2608,0.3658,-0.4561,0.9777,0.4758
4,food_confidence_slope,derived_category_slope,0.0219,0.0445,-0.0653,0.1091,0.6227
5,nonfood_confidence_slope,derived_category_slope,0.2804,0.1037,0.0771,0.4837,0.0069


The interaction coefficients are unchanged by construction because this check replaces only the covariance estimator. Relative to country-month clustered Phase 2 uncertainty, Driscoll–Kraay standard errors rise from 0.229 to 0.410 for inflation×non-food and from 0.073 to 0.115 for confidence×non-food. Both 95% intervals still exclude zero, so the category contrasts survive, but with weaker precision. The derived pattern remains: food growth is negatively associated with inflation, while non-food growth is positively associated with confidence.

## Phase 2.5 takeaway

The narrow story that survives is **category asymmetry**, especially the negative food–inflation association and the food/non-food contrasts. The broader claim that confidence independently tracks total or non-food retail growth is sensitive to controlling for common monthly shocks and should not be presented as generally robust. Phase 3 may proceed only if it carries these qualifications forward, retains associational language, and treats the three-country time-fixed-effects evidence as limited.